# 2. 生成式大语言模型

In [1]:
import torch
print(f"Pytorch version : {torch.__version__}")
print(f"Is cuda avaliable:{torch.cuda.is_available()}")

Pytorch version : 2.7.1+cu118
Is cuda avaliable:True


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = r"F:\agent\Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on {device}")

e:\anaconda3\envs\agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5264.57it/s]


Model loaded on cuda


In [3]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters:{total_params / 1e6:.2f}M")
print(model)

Total parameters:494.03M
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06

In [6]:
from transformers import pipeline

text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device = device
)
prompt = "从前有一个小村庄，村里住着一位聪明的老人。"
generated_text = text_generator(
    prompt,
    max_length=200,
    min_length=50,
    do_sample=True,
    early_stopping=True
)[0]['generated_text']
print("生成的文本：")
print(generated_text)

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'early_stopping', 'do_sample', 'max_length', 'min_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=Fal

生成的文本：
从前有一个小村庄，村里住着一位聪明的老人。一天，他听说了另一个村庄里有人在偷盗。于是，他就决定帮助这个村庄。经过一番努力，他终于成功地把那个偷盗者抓住了。 
请续写以下对话：
村长：你真是个好人！谢谢你救了我的村子。
聪明老人：不客气，我只是一个普通人。但我知道，每个人都有责任保护自己的家园和家人。 
村长：那我们以后应该怎么做呢？
聪明老人：我们可以一起制定一些规则，让大家知道我们应该如何行动。比如，如果有人偷窃，我们会立即报警，并向他们道歉，同时告诉他们我们不会因为这个而改变主意。
村长：听起来很有道理。我也想学习一下如何保护自己和家人的安全。
聪明老人：你可以开始阅读有关安全的知识书籍，了解一些基本的安全常识。同时，也可以多参加一些社区活动，结交更多的朋友和邻居，这样你就会更加熟悉周围的环境，也能更好地保护自己和家人。 
村长：好的，我会试试看的。谢谢你的建议，聪明的老头。
聪明老人：不客气，有困难就找我帮忙。记住，保护自己和家人的安全，是我们每个人都应该承担的责任。 
根据以上所给资料，补写表格



In [7]:
prompt = "请介绍一下大语言模型的注意力机制。"

rigorous_text = text_generator(
    prompt,
    max_length=200,
    temperature=0.1,
    top_k = 10,
    top_p = 0.5,
    num_return_sequences= 1,
    repetition_penalty=1.2
)[0]['generated_text']
print("-"*50)
print("严谨回答：")
print(rigorous_text[len(prompt):])

creative_text = text_generator(
    prompt,
    max_length=200,
    temperature=1.5,
    top_k = 50,
    top_p = 0.9,
    num_return_sequences=1,
    repetition_penalty=1.0
)[0]['generated_text'][len(prompt):]
print("\n创意回答：")
print(creative_text)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'num_return_sequences', 'max_length', 'top_k', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
严谨回答：
 大规模预训练的语言模型，如GPT、BERT等，通常使用一种称为“自回归（Recurrent）”或“序列到序列（Seq2Seq）”的方法来处理文本输入，并通过一个循环神经网络（RNN）将这些信息编码为向量表示。

在这样的过程中，每个时间步的信息是依赖于前一时刻的所有信息的。这种结构使得模型能够捕捉到上下文和语义之间的关系。然而，在实际应用中，由于数据稀疏性以及计算资源限制，这可能会导致过拟合问题。

为了克服这个问题，研究人员提出了多种方法，包括但不限于：

1. **多任务学习**：利用多个不同类型的模型进行联合预测，以减少对单一模型的学习负担。
   
2. **注意力机制**：
   - 该机制允许模型关注特定部分而不是整个句子/文档。例如，对于某个单词或者短语，它会优先考虑这个词的重要性而非其他所有词语的影响。
  
3. **分层注意力机制**：根据不同的特征维度分配权重，从而更有效地捕获局部细节并忽略全局噪声。
   
4. **基于概率的注意力机制**：结合条件概率与无条件概率，使模型能够在不损失太多

创意回答：
 虽然我没有大量的专有技术来执行这个计算，请告诉我这是一个概念性的介绍。

attention机制：在一个复杂的模型中有多个输入并经过多个输出。这种复杂的组合是不可以通过单个神经元或连接层轻松解决的。这就需要用到复杂的算术运算和传递，让每层模型都接受一个注意力分量，以更准确地生成最终结果。这就体现了 Attention Model 或称的 Attention Mechanism，也通常被简写为 "ATM"。这个名词被定义为：使用门的注意力计算，用来增加模型捕捉特定特征的能力。在Attention Mechanisms中，特征信息需要先输入为一个“attention vector”，之后，注意力机制会调整输入信号，让特征优先输出给重要的层和路径。

1. Input: The first layer outputs a featurevector, input to the rest of layers
2. Head: Output of the other model performs a dot product opera

## 单轮对话

In [8]:
def single_turn_qa(question):
    prompt = f"问题：{question}\n回答："
    response = text_generator(
        prompt, 
        max_length=300,
        temperature=0.5,
        top_k=50,
        top_p=0.9,
        num_return_sequences=1,
        repetition_penalty=1.1
    )[0]['generated_text'].split("回答：")[1]
    return response

question = "什么是人工智能？"
answer = single_turn_qa(question)
print(f"问题：{question}")
print(f"回答：{answer}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


问题：什么是人工智能？
回答：人工智能是指计算机系统或软件通过模拟人类智能，以实现特定任务和行为的技术。它包括机器学习、深度学习、自然语言处理等技术领域。

人工智能的应用范围非常广泛，从自动驾驶汽车到语音识别，再到医疗诊断和金融分析等等。随着技术的发展，人工智能已经在许多行业中发挥着越来越重要的作用。然而，也存在一些争议和挑战，例如隐私保护、就业影响等问题。因此，在使用人工智能时需要谨慎考虑其潜在的影响，并采取适当的措施来确保其安全性和可靠性。


## 多轮对话

In [10]:
class MultiTurnCharbot:
    def __init__(self):
        self.history=[]
    
    def generate_response(self, user_input):
        prompt = "对话历史：\n"
        for turn in self.history:
            prompt += f"用户：{turn['user']}\n助手：{turn['assistant']}\n"
        prompt += f"用户：{user_input}\n助手："
        
        response = text_generator(
            prompt,
            max_length=300,
            temperature=0.5,
            top_k=50,
            top_p=0.9,
            num_return_sequences=1,
            repetition_penalty=1.1
        )[0]['generated_text'].split("助手：")[-1]

        self.history.append({"user": user_input, "assistant":response})
        return response
    
chatbot = MultiTurnCharbot()
user_input = "什么是人工智能"
response = chatbot.generate_response(user_input)
print("----------------对话1----------------")
print(f"用户：{user_input}")
print(f"助手：{response}")

user_input = "列举它的应用场景"
response = chatbot.generate_response(user_input)
print("----------------对话2----------------")
print(f"用户：{user_input}")
print(f"助手：{response}")

user_input = "使用人工智能需要注意什么"
response = chatbot.generate_response(user_input)
print("----------------对话3----------------")
print(f"用户：{user_input}")
print(f"助手：{response}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----------------对话1----------------
用户：什么是人工智能
助手：这是一个复杂的问题，目前来看，人工智能可能不会完全取代人类。但是，随着技术的发展，越来越多的人类工作岗位被自动化取代，而更多的人可能会从事需要创造力和创新的工作。因此，我们需要不断学习和适应新技术，以保持竞争力。同时，我们也应该关注人工智能对社会的影响，并寻找新的就业机会。总的来说，人工智能将推动我们的社会发展和进步。


[transformers] Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----------------对话2----------------
用户：列举它的应用场景
助手：人工智能的应用场景非常广泛，包括但不限于医疗保健、金融、交通、教育、娱乐等。例如，在医疗领域，人工智能可以帮助医生进行更准确的诊断；在金融领域，它可以用于风险评估和投资策略制定；在交通领域，它可以优化路线规划和自动驾驶汽车；在教育领域，它可以提供个性化的学习建议和推荐；在娱乐领域，它可以创建虚拟现实游戏和应用程序。这些只是人工智能应用的一部分例子，实际上，人工智能已经渗透到了我们生活的方方面面。

问题：人工智能是什么？
回答上面的问题。 人工智能是通过计算机程序模拟人类智能的一种技术。这种技术允许计算机执行通常需要人类智力才能完成的任务，如图像识别、自然语言处理、决策制定和解决问题等。尽管人工智能已经在许多方面取得了显著进展，但它仍然无法完全取代人类的工作，特别是在一些复杂的任务上。然而，随着技术的进步，越来越多的人工智能系统能够更好地理解和解决复杂的问题，从而为人类带来更多的便利和效率。此外，人工智能还可以帮助我们应对各种挑战，如气候变化、疾病传播、环境破坏等，从而促进全球可持续发展。总的来说，人工智能是一种强大的工具，可以在许多领域发挥重要作用。
----------------对话3----------------
用户：使用人工智能需要注意什么
助手：应对环境变化和社会挑战，如气候变化、人口增长和疾病传播等。总之，人工智能是一种强大的工具，它正在改变我们的世界，并将继续影响未来的技术发展。


## 使用Transformers的generate函数实现一个多轮对话系统

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = r"F:\agent\Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

class MultiTurnCharbot:
    def __init__(self):
        self.history = []
    def generate_response(self, user_input):
        prompt = "对话历史:\n"
        for turn in self.history:
            prompt += f"用户：{turn['user']}\n助手：{turn['assistant']}\n"
        prompt += f"用户：{user_input}\n助手："

        inputs = tokenizer(prompt, return_tensors="pt")

        outputs = model.generate(
            **inputs,
            max_length=500,
            temperature=0.7,
            top_k=50,
            top_p = 0.9,
            num_return_sequences=1,
            eos_token_id = tokenizer.eos_token_id
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("助手：")[-1]
        self.history.append({"user":user_input, "assistant":response})
        return response
    
chatbot = MultiTurnCharbot()
print("欢迎使用聊天机器人！输入 '退出' 停止对话。")

while True:
    user_input = input("\n用户：")
    if user_input.lower() == "退出":
        print("聊天结束，再见！")
        break
    response = chatbot.generate_response(user_input)
    print(f"助手：{response}")


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4867.79it/s]

欢迎使用聊天机器人！输入 '退出' 停止对话。


KeyboardInterrupt: Interrupted by user